In [1]:
# =================================================================================================
# NHANES 2009-2010
# Sleep Disorders Questionnaire
#
# File:
#   SLQ_F.xpt
#
# Purpose:
#   Convert NHANES SLQ_F XPT -> fidelity-preserving corrected CSV.
#
# IMPORTANT:
#
#   SLQ_F is NHANES 2009-2010, NOT 2017-2018.
#
#   Attached SLQ_F contains:
#
#       SEQN
#       SLD010H
#       SLQ050
#       SLQ060
#
#
# Fidelity findings from attached XPT:
#
#   Rows                         = 6,889
#   Columns                      = 4
#   SEQN range                   = 51624-62160
#   Duplicate SEQN               = 0
#   Missing SEQN                 = 0
#
#   SAS XPORT tiny-zero artifact = 0
#   True numeric zeros           = 0
#
#   Genuine decimal variables    = NONE
#
#
# Fidelity rules:
#
#   - Remove unnecessary ".0" from SEQN
#   - Do NOT replace any values with zero
#   - Do NOT round
#   - Do NOT use float_format=
#   - Preserve missing positions
#   - Validate against official CDC frequencies
#   - Convert only proven whole-number/code fields to nullable Int64
#   - Verify XPT -> CSV round-trip fidelity
#
#
# General NHANES pipeline rule:
#
#   Genuine Dietary / Lab / Examination decimal variables in other files
#   must remain AS IS.
#
#   This particular SLQ_F file contains no genuine decimal variables.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)


XPT_FILE = (
    DATA_DIR
    / "SLQ_F.xpt"
)


CSV_FILE = (
    DATA_DIR
    / "SLQ_F_corrected.csv"
)


# =================================================================================================
# 3. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Keep untouched source for fidelity comparison.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2009-2010 SLQ_F XPT -> CSV"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Columns:",
    df.columns.tolist()
)


# =================================================================================================
# 4. EXPECTED OFFICIAL STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "SLD010H",

    "SLQ050",

    "SLQ060"

]


EXPECTED_ROWS = 6889


if (
    df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "SLQ_F column structure differs from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected SLQ_F row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: Structure = 6,889 rows x 4 columns."
)


# =================================================================================================
# 5. VERIFY ALL VARIABLES LOADED NUMERICALLY
# =================================================================================================

numeric_columns = (

    df.select_dtypes(
        include=[np.number]
    )

    .columns

    .tolist()

)


if (
    numeric_columns
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Expected all four SLQ_F variables "
        "to load numerically from XPT."
    )


print(
    "PASS: All 4 variables loaded numerically."
)


# =================================================================================================
# 6. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (

    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- XPORT TINY-ZERO CHECK ---"
)


print(
    "Tiny values by variable:"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny XPORT values:",
    tiny_total
)


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT values were found.\n"
        "This attached SLQ_F was validated with zero tiny artifacts."
    )


print(
    "PASS: No SAS XPORT tiny-zero artifacts."
)


# =================================================================================================
# 7. TRUE NUMERIC ZERO CHECK
# =================================================================================================

zero_counts = (

    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()

)


zero_total = int(
    zero_counts.sum()
)


print(
    "\n--- TRUE ZERO CHECK ---"
)


print(
    zero_counts.to_string()
)


print(
    "Total numeric zeros:",
    zero_total
)


if zero_total != 0:

    raise ValueError(
        "Unexpected zero values found in SLQ_F."
    )


print(
    "PASS: No legitimate zero-coded variables exist "
    "in this SLQ_F release."
)


# =================================================================================================
# 8. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# Diagnostic only.
#
# np.round() is used only for comparison.
# It does NOT modify source values.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (

        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        fractional_count = 0


    else:

        fractional_count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[col] = (
        fractional_count
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional values"
    )


fractional_variables = {

    col: count

    for col, count in fractional_counts.items()

    if count > 0

}


if fractional_variables:

    raise ValueError(
        "Unexpected fractional variables found:\n"
        + str(
            fractional_variables
        )
    )


print(
    "PASS: No genuine decimal variables in SLQ_F."
)


# =================================================================================================
# 9. FIELD TYPE PLAN
# =================================================================================================

#
# All variables are proven whole-number/code fields.
#

INTEGER_COLUMNS = [

    "SEQN",

    "SLD010H",

    "SLQ050",

    "SLQ060"

]


DECIMAL_COLUMNS = []


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# =================================================================================================
# 10. SEQN VALIDATION
# =================================================================================================

print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 51624:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 62160:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if (
    df["SEQN"]
    .isna()
    .any()
):

    raise ValueError(
        "SEQN contains missing values."
    )


if (
    df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN range 51624-62160 validated."
)


# =================================================================================================
# 11. CDC CHECK - SLD010H
# =================================================================================================

#
# CDC official counts:
#
# 2-11 hours        = 6,835
# 12+ hours         =    39
# Refused           =     0
# Don't know (99)   =     8
# Missing           =     7
#
# Total             = 6,889
#

sld010h_2_to_11 = int(

    df["SLD010H"]
    .between(
        2,
        11
    )
    .sum()

)


sld010h_12 = int(

    df["SLD010H"]
    .eq(12)
    .sum()

)


sld010h_77 = int(

    df["SLD010H"]
    .eq(77)
    .sum()

)


sld010h_99 = int(

    df["SLD010H"]
    .eq(99)
    .sum()

)


sld010h_missing = int(

    df["SLD010H"]
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: SLD010H ---"
)


print(
    "2-11 hours:",
    sld010h_2_to_11
)


print(
    "12+ hours:",
    sld010h_12
)


print(
    "77 Refused:",
    sld010h_77
)


print(
    "99 Don't know:",
    sld010h_99
)


print(
    "Missing:",
    sld010h_missing
)


if sld010h_2_to_11 != 6835:

    raise ValueError(
        "SLD010H range 2-11 differs from CDC."
    )


if sld010h_12 != 39:

    raise ValueError(
        "SLD010H=12 differs from CDC."
    )


if sld010h_77 != 0:

    raise ValueError(
        "SLD010H Refused count differs from CDC."
    )


if sld010h_99 != 8:

    raise ValueError(
        "SLD010H Don't know count differs from CDC."
    )


if sld010h_missing != 7:

    raise ValueError(
        "SLD010H missing count differs from CDC."
    )


print(
    "PASS: SLD010H matches CDC."
)


# =================================================================================================
# 12. CDC CHECK - SLQ050
# =================================================================================================

#
# CDC:
#
# 1 = Yes          1,601
# 2 = No           5,285
# 7 = Refused          0
# 9 = Don't know       3
# Missing              0
#

SLQ050_EXPECTED = {

    1.0: 1601,

    2.0: 5285,

    9.0: 3

}


slq050_actual = (

    df["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


slq050_missing = int(

    df["SLQ050"]
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: SLQ050 ---"
)


print(
    "Observed:",
    slq050_actual
)


print(
    "Missing:",
    slq050_missing
)


if slq050_actual != SLQ050_EXPECTED:

    raise ValueError(
        "SLQ050 distribution differs from CDC.\n\n"
        f"Expected:\n{SLQ050_EXPECTED}\n\n"
        f"Actual:\n{slq050_actual}"
    )


if slq050_missing != 0:

    raise ValueError(
        "SLQ050 missing count differs from CDC."
    )


print(
    "PASS: SLQ050 matches CDC."
)


# =================================================================================================
# 13. CDC CHECK - SLQ060
# =================================================================================================

#
# CDC:
#
# 1 = Yes            477
# 2 = No           6,403
# 7 = Refused          0
# 9 = Don't know       9
# Missing              0
#

SLQ060_EXPECTED = {

    1.0: 477,

    2.0: 6403,

    9.0: 9

}


slq060_actual = (

    df["SLQ060"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


slq060_missing = int(

    df["SLQ060"]
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: SLQ060 ---"
)


print(
    "Observed:",
    slq060_actual
)


print(
    "Missing:",
    slq060_missing
)


if slq060_actual != SLQ060_EXPECTED:

    raise ValueError(
        "SLQ060 distribution differs from CDC.\n\n"
        f"Expected:\n{SLQ060_EXPECTED}\n\n"
        f"Actual:\n{slq060_actual}"
    )


if slq060_missing != 0:

    raise ValueError(
        "SLQ060 missing count differs from CDC."
    )


print(
    "PASS: SLQ060 matches CDC."
)


# =================================================================================================
# 14. VERIFY NO SOURCE VALUE NEEDS CORRECTION
# =================================================================================================

#
# Unlike SLQ_D and SLQ_E:
#
# SLQ_F contains NO validated tiny-zero artifacts.
#
# Therefore the working dataframe should still be numerically
# identical to the original pandas-decoded XPT at this point.
#

source_value_failures = []


for col in EXPECTED_COLUMNS:

    original_values = (

        source_df[col]
        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]
        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        original_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_value_failures.append(
            col
        )


if source_value_failures:

    raise ValueError(
        "Unexpected source values changed:\n"
        + str(
            source_value_failures
        )
    )


print(
    "PASS: No source numeric values changed."
)


# =================================================================================================
# 15. MISSING-POSITION FIDELITY
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    original_missing = (

        source_df[col]
        .isna()
        .to_numpy()

    )


    current_missing = (

        df[col]
        .isna()
        .to_numpy()

    )


    missing_position_errors += int(

        (
            original_missing
            !=
            current_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions unchanged."
)


# =================================================================================================
# 16. CONVERT PROVEN WHOLE-NUMBER FIELDS TO NULLABLE Int64
# =================================================================================================

#
# This converts:
#
#     SEQN = 51624.0
#
# to:
#
#     SEQN = 51624
#
#
# It also prevents code values such as:
#
#     SLQ050 = 1.0
#
# from being written with unnecessary ".0".
#
#
# No true decimal field exists in this particular source.
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All four proven whole-number fields "
    "converted to nullable Int64."
)


# =================================================================================================
# 17. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        source_df[col]
        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 18. EXPORT CORRECTED CSV
# =================================================================================================

#
# IMPORTANT:
#
# No rounding.
#
# No float_format=.
#
# This is also the correct rule for your broader NHANES
# Dietary/Lab/Examination processing where decimals exist.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 19. RE-READ FINAL CSV WITH ROUND-TRIP PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 20. EXACT NUMERIC ROUND-TRIP CHECK
# =================================================================================================

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        source_df[col]
        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All four variables round-trip exactly."
)


# =================================================================================================
# 21. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 22. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' values."
)


# =================================================================================================
# 23. FINAL SEQN CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN contains '.0'."
    )


if (
    ~raw_csv["SEQN"]
    .str.fullmatch(r"\d+")
).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


seqn_numeric = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if seqn_numeric.min() != 51624:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if seqn_numeric.max() != 62160:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN is clean."
)


# =================================================================================================
# 24. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts exist in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 25. FINAL TRUE-ZERO CHECK
# =================================================================================================

final_zero_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_zero_total += int(

        values
        .eq(0)
        .sum()

    )


if final_zero_total != 0:

    raise ValueError(
        "Unexpected zero values exist in final CSV."
    )


print(
    "PASS: Final CSV contains no unexpected zeros."
)


# =================================================================================================
# 26. SCIENTIFIC NOTATION CHECK
# =================================================================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_errors[
            col
        ] = count


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_errors
        )
    )


print(
    "PASS: No scientific notation."
)


# =================================================================================================
# 27. FINAL MISSING-POSITION CHECK
# =================================================================================================

final_missing_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]
        .isna()
        .to_numpy()

    )


    csv_missing = (

        raw_csv[col]
        .eq("")
        .to_numpy()

    )


    final_missing_errors += int(

        (
            source_missing
            !=
            csv_missing
        )

        .sum()

    )


if final_missing_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from XPT."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 28. FINAL CDC FREQUENCY RECHECK
# =================================================================================================

#
# Read final CSV numerically for validation only.
#

final_numeric = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


#
# SLD010H
#

if int(
    final_numeric["SLD010H"]
    .between(2, 11)
    .sum()
) != 6835:

    raise ValueError(
        "Final SLD010H 2-11 count differs."
    )


if int(
    final_numeric["SLD010H"]
    .eq(12)
    .sum()
) != 39:

    raise ValueError(
        "Final SLD010H=12 count differs."
    )


if int(
    final_numeric["SLD010H"]
    .eq(99)
    .sum()
) != 8:

    raise ValueError(
        "Final SLD010H=99 count differs."
    )


if int(
    final_numeric["SLD010H"]
    .isna()
    .sum()
) != 7:

    raise ValueError(
        "Final SLD010H missing count differs."
    )


#
# SLQ050
#

final_slq050 = (

    final_numeric["SLQ050"]

    .value_counts()

    .sort_index()

    .to_dict()

)


if final_slq050 != {

    1: 1601,

    2: 5285,

    9: 3

}:

    raise ValueError(
        "Final SLQ050 distribution differs."
    )


#
# SLQ060
#

final_slq060 = (

    final_numeric["SLQ060"]

    .value_counts()

    .sort_index()

    .to_dict()

)


if final_slq060 != {

    1: 477,

    2: 6403,

    9: 9

}:

    raise ValueError(
        "Final SLQ060 distribution differs."
    )


print(
    "PASS: Final CSV frequencies match CDC."
)


# =================================================================================================
# 29. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2009-2010 SLQ_F CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2009-2010"
)


print(
    ""
)


print(
    "Component:"
)


print(
    "  Questionnaire - Sleep Disorders"
)


print(
    ""
)


print(
    "Eligible population:"
)


print(
    "  Age 16 years and older"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    ""
)


print(
    "Fields:"
)


print(
    "  SEQN"
)


print(
    "  SLD010H"
)


print(
    "  SLQ050"
)


print(
    "  SLQ060"
)


print(
    ""
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Validated zero restorations:",
    0
)


print(
    "True zero values:",
    0
)


print(
    ""
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Unexpected value changes: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Round-trip numeric failures: 0"
)


print(
    ""
)


print(
    "CDC validation:"
)


print(
    "  SLD010H: PASS"
)


print(
    "  SLQ050:  PASS"
)


print(
    "  SLQ060:  PASS"
)


print(
    "=" * 110
)

NHANES 2009-2010 SLQ_F XPT -> CSV
Rows: 6,889
Columns: 4
Columns: ['SEQN', 'SLD010H', 'SLQ050', 'SLQ060']
PASS: Structure = 6,889 rows x 4 columns.
PASS: All 4 variables loaded numerically.

--- XPORT TINY-ZERO CHECK ---
Tiny values by variable:
SEQN       0
SLD010H    0
SLQ050     0
SLQ060     0

Total tiny XPORT values: 0
PASS: No SAS XPORT tiny-zero artifacts.

--- TRUE ZERO CHECK ---
SEQN       0
SLD010H    0
SLQ050     0
SLQ060     0
Total numeric zeros: 0
PASS: No legitimate zero-coded variables exist in this SLQ_F release.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional values
SLD010H: 0 fractional values
SLQ050: 0 fractional values
SLQ060: 0 fractional values
PASS: No genuine decimal variables in SLQ_F.

--- FIELD TYPE PLAN ---
Whole-number/code fields: ['SEQN', 'SLD010H', 'SLQ050', 'SLQ060']
Genuine decimal fields: []

--- SEQN SOURCE CHECK ---
Minimum: 51624.0
Maximum: 62160.0
Missing: 0
Duplicates: 0
PASS: SEQN range 51624-62160 validated.

--- CDC CHECK: SLD010H ---
2-11